# Task 3 ? PakFreight Data Detective

This notebook documents the raw-file audit, stovepipe totals, formula-based staging, reconciliation, lineage of 578.65 tonnes, and the Kimball bus matrix. Source files and the student workbook are never edited.

In [1]:
from pathlib import Path
import pandas as pd
from openpyxl import load_workbook
ROOT=Path.cwd()
if not (ROOT/"marine_shipments.csv").exists(): ROOT=Path(r"D:\University\ML Lab\Week02")

## 1. Raw-file audit

Audit before cleaning: units, dates, identifiers, casing/whitespace, duplicate rows, blank rows, non-data rows, and missing values.

In [2]:
marine=pd.read_csv(ROOT/"marine_shipments.csv")
road=pd.read_csv(ROOT/"road_shipments.csv")
rail_raw=pd.read_excel(ROOT/"rail_shipments.xlsx",header=None)
rail=pd.read_excel(ROOT/"rail_shipments.xlsx",skiprows=2)
print("Marine",marine.shape,"Rail raw",rail_raw.shape,"Rail data",rail.shape,"Road",road.shape)
print("Rail unit/title rows:",rail_raw.iloc[0,0],"/",rail_raw.iloc[1,0])
print("Marine date example:",marine.loc[2,"Ship_Date"])
print("Status examples:",repr(marine.Status.iloc[1]),repr(marine.Status.iloc[2]),repr(road.status.iloc[2]))
print("Road blank rows:",int(road.isna().all(axis=1).sum()))
print("Road TOTAL rows:",int(road.trip_id.eq("TOTAL").sum()))
print("Duplicate road IDs:",road.loc[road.trip_id.notna()&road.trip_id.duplicated(False),"trip_id"].tolist())
print("Missing values:\nMarine",marine.isna().sum().to_dict(),"\nRail",rail.isna().sum().to_dict(),"\nRoad",road.isna().sum().to_dict())

Marine (12, 7) Rail raw (13, 7) Rail data (10, 7) Road (15, 7)
Rail unit/title rows: Pakistan Railways Freight — Consignment Register / Period: February 2026   |   Weights in TONS
Marine date example: 03/02/2026
Status examples: 'DELIVERED ' 'IN TRANSIT' 'in transit '
Road blank rows: 1
Road TOTAL rows: 1
Duplicate road IDs: ['rd-505', 'rd-505']
Missing values:
Marine {'Shipment_ID': 0, 'Ship_Date': 0, 'Origin_Port': 0, 'Dest_Port': 0, 'Weight_KG': 0, 'Cost_PKR': 0, 'Status': 0} 
Rail {'Consignment No': 0, 'Dispatch Date': 0, 'From Station': 0, 'To Station': 0, 'Weight (Tons)': 0, 'Freight Charges (PKR)': 0, 'Status': 0} 
Road {'trip_id': 1, 'trip_date': 2, 'from_city': 2, 'to_city': 2, 'weight_kg': 1, 'cost_pkr': 1, 'status': 2}


### Logged problems

| File/field/example | Problem and reporting effect | Formula/staging fix |
|---|---|---|
| Marine `Ship_Date=03/02/2026` | Text dd/mm/yyyy can be swapped or fail under another locale | Build `DATE(year,month,day)` |
| Marine `Status=DELIVERED ` and `IN TRANSIT` | Whitespace/case fragments statuses | `TRIM` + `PROPER` |
| Rail first two rows | Title/non-data rows shift the header | Remove top two rows |
| Rail `Weight (Tons)=32.5` | Reading as kg understates weight 1,000? | Multiply by 1,000 |
| Rail `Consignment No=45012` | Missing `RL-` identifier prefix breaks joins | `="RL-"&A5` |
| Rail statuses | Mixed case/whitespace fragments status counts | `TRIM` + `PROPER` |
| Road `rd-505` twice | Duplicate shipment double-counts weight | Keep first occurrence |
| Road blank row | Missing fact can inflate/null-load records | Mark N: Blank row |
| Road `TOTAL` | Summary row is not a shipment; double-counts | Mark N: Non-data TOTAL row |
| Road lower-case IDs/statuses | Inconsistent joins and grouping | `UPPER`, `TRIM`, `PROPER` |

## 2. Stovepipe totals

In [3]:
marine_kg=marine.Weight_KG.sum(); rail_tons=rail["Weight (Tons)"].sum(); rail_kg=rail_tons*1000
real=road.loc[road.trip_id.notna()&road.trip_id.ne("TOTAL")]
road_true=real.drop_duplicates("trip_id").weight_kg.sum(); road_dup=real.weight_kg.sum(); road_total=road.weight_kg.sum(); dup_weight=road.loc[road.trip_id.eq("rd-505"),"weight_kg"].iloc[1]
operations=marine_kg+rail_kg+road_total-dup_weight; finance=marine_kg+rail_tons+road_true; sales=marine_kg+rail_kg+road_dup; truth=marine_kg+rail_kg+road_true
print("Operations: 206,500 + 295,250 + 83,000 - 6,100 =",operations,"kg = 661.65 t")
print("Finance: 206,500 + 295.25 + 76,900 =",finance,"kg = 283.70 t")
print("Sales: 206,500 + 295,250 + 83,000 =",sales,"kg = 584.75 t")
print("Truth: 206,500 + 295,250 + 76,900 =",truth,"kg = 578.65 t")
assert [round(x/1000,2) for x in [operations,finance,sales,truth]]==[661.65,283.70,584.75,578.65]

Operations: 206,500 + 295,250 + 83,000 - 6,100 = 661650.0 kg = 661.65 t
Finance: 206,500 + 295.25 + 76,900 = 283695.25 kg = 283.70 t
Sales: 206,500 + 295,250 + 83,000 = 584750.0 kg = 584.75 t
Truth: 206,500 + 295,250 + 76,900 = 578650.0 kg = 578.65 t


All three departments believed they were right because each applied a consistent local rule to its own interpretation of the source schema. Without a shared staging contract and reconciliation control, a summary row, unit header, or duplicate could appear plausible inside an isolated mini-warehouse.

## 3?4. Formula staging and reconciliation

In [4]:
wb=load_workbook(ROOT/"PakFreight_Activity_COMPLETED.xlsx",data_only=False)
checks={"Marine date":wb["T3a Marine_Stg"]["I7"].value,"Rail kg":wb["T3b Rail_Stg"]["L5"].value,"Road keep":wb["T3c Road_Stg"]["P10"].value,"Road reason":wb["T3c Road_Stg"]["Q10"].value}
print(checks); assert all(str(v).startswith("=") for v in checks.values())
reconcile=pd.DataFrame({"Source":["Marine","Rail","Road","Total"],"Raw records":[12,10,15,37],"Removed":[0,0,3,3],"Loaded":[12,10,12,34],"Weight kg":[206500,295250,76900,578650]})
print(reconcile.to_string(index=False)); assert reconcile.iloc[-1]["Weight kg"]==578650

{'Marine date': '=DATE(VALUE(RIGHT(B7,4)),VALUE(MID(B7,4,2)),VALUE(LEFT(B7,2)))', 'Rail kg': '=E5*1000', 'Road keep': '=IF(A10="","N",IF(UPPER(A10)="TOTAL","N",IF(COUNTIF($A$5:A10,A10)>1,"N","Y")))', 'Road reason': '=IF(P10="Y","",IF(A10="","Blank row",IF(UPPER(A10)="TOTAL","Non-data TOTAL row","Duplicate shipment ID")))'}
Source  Raw records  Removed  Loaded  Weight kg
Marine           12        0      12     206500
  Rail           10        0      10     295250
  Road           15        3      12      76900
 Total           37        3      34     578650


Road removals are three different rows: the second `rd-505` is a duplicate, row 18 is entirely blank, and row 19 `TOTAL` is a non-data summary row. Add a staging gate that rejects `TOTAL` and any `Keep? <> Y`, and fails refresh unless raw ? removed = loaded and total weight equals 578,650 kg.

## 5. Lineage of 578.65 tonnes

1. Read `marine_shipments.csv` (12), `rail_shipments.xlsx` after two title rows (10), and `road_shipments.csv` (15).
2. Stage with parsed dates, standardized identifiers/statuses, Rail tons ? 1,000, and a `SourceSystem` column.
3. Road keeps the first `rd-505`; rejects the blank row and `TOTAL`: 12 loaded.
4. Union conformed staging: 12 + 10 + 12 = 34 shipment facts.
5. Sum canonical WeightKG: 206,500 + 295,250 + 76,900 = 578,650 kg.
6. Presentation converts once: 578,650 / 1,000 = **578.65 t**.

Every row remains traceable by ShipmentID and SourceSystem; source files are unchanged.

## 6. Kimball bus matrix

In [5]:
bus=pd.DataFrame({"Date":["X","X","X","X","X"],"Customer":["X","X","","X","X"],"Route":["X","X","","X","X"],"Vehicle":["X","X","X","X",""],"Driver":["","","X","X",""],"Station/Port":["X","X","X","X","X"],"Commodity":["X","X","","X","X"]},index=["Shipments","Billing & Invoicing","Fleet Maintenance","Driver Trips & Hours","Customer Complaints"])
print(bus.to_string())

                     Date Customer Route Vehicle Driver Station/Port Commodity
Shipments               X        X     X       X                   X         X
Billing & Invoicing     X        X     X       X                   X         X
Fleet Maintenance       X                      X      X            X          
Driver Trips & Hours    X        X     X       X      X            X         X
Customer Complaints     X        X     X                           X         X


Date is used by every process, so it is the first conformed dimension. Finance, Operations, and Sales must agree on shipment identity, date, canonical weight unit, and route. Kimball is bottom-up in delivery, not unplanned: the bus matrix defines shared dimensions while stars are built process by process. Build Shipments first for the COO because it establishes the canonical shipment and WeightKG facts Billing can reuse.

## Validation summary

Marine 12/0/12/206,500 kg; Rail 10/0/10/295,250 kg; Road 15/3/12/76,900 kg; Total 37/3/34/578,650 kg = 578.65 t. The output workbook is a separate copy: `PakFreight_Activity_COMPLETED.xlsx`.